# IndoBERTweet + Weighted Loss — Sentimen Isu Ijazah Jokowi
### Dataset **v1audited** (label LLM + audit, 3 kelas seimbang)

Notebook ini mem-*fine-tune* **IndoBERTweet** untuk mengklasifikasikan sikap komentar YouTube
terhadap narasi *"dugaan ijazah Jokowi palsu"* → **Negatif / Netral / Positif**.

**Kenapa IndoBERTweet?** `indolem/indobertweet-base-uncased` di-*pretrain* pada **Twitter
Indonesia** (bahasa medsos/alay/singkatan) — lebih cocok untuk komentar YouTube dibanding
IndoBERT biasa (Wikipedia/berita).

**Kenapa Weighted Loss?** Setelah audit, kelas sedikit tak seimbang; *weighted cross-entropy*
memberi bobot lebih besar ke kelas minoritas agar tak terabaikan.

| Komponen | Nilai |
|---|---|
| Model | `indolem/indobertweet-base-uncased` |
| Epoch | 6 (+ early stopping, ambil checkpoint val terbaik) |
| Learning rate | `2e-5` |
| Batch | 16 train / 32 eval |
| Max token | 128 |
| Loss | weighted cross-entropy |
| Split | 70/20/10, seed 42 — **identik** SVM & IndoBERT (adil) |

**Alur data:** teks fitur `bert` (lowercase, buang URL/emoji; tanda baca & imbuhan dipertahankan)
dari MongoDB; **label** dari `balanced_3000_v1audited.csv`.

> **Cara pakai:** Runtime → Change runtime type → **T4 GPU**, lalu **Run all**.
> Saat diminta isi `GITHUB_PAT` (clone repo) & `MONGO_URI`.

## 0. Setup — cek GPU & pasang dependency

In [ ]:
!nvidia-smi -L || echo '⚠️  Tidak ada GPU — aktifkan: Runtime > Change runtime type > T4 GPU'
%pip install -q "transformers>=4.40" torch "pymongo[srv]" dnspython certifi scikit-learn matplotlib pandas numpy

## 1. Ambil kode & data
Clone repo (untuk file label `balanced_3000_v1audited.csv` & artefak pembanding) + koneksi
MongoDB (teks fitur `bert`). Token dari *Colab Secrets* (🔑) bila ada, atau ketik manual; token
tidak ditampilkan.

In [ ]:
import os, subprocess, shutil, pathlib

def ambil_rahasia(nama):
    """env var -> Colab Secrets -> input manual."""
    v = os.environ.get(nama, '')
    if not v:
        try:
            from google.colab import userdata
            v = userdata.get(nama) or ''
        except Exception:
            v = ''
    if not v:
        from getpass import getpass
        v = getpass(f'{nama}: ')
    return v

GITHUB_PAT = ambil_rahasia('GITHUB_PAT')
os.environ['MONGO_URI'] = ambil_rahasia('MONGO_URI')

REPO = '/content/jokowi_sentiment_project'
if pathlib.Path(REPO).exists():
    shutil.rmtree(REPO)
url = f'https://{GITHUB_PAT}@github.com/Arachnoida/jokowi_sentiment_project.git'
hasil = subprocess.run(['git', 'clone', '--depth', '1', url, REPO],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('✅ Repo ter-clone' if hasil.returncode == 0 else '❌ Clone gagal (cek PAT):')
print('\n'.join(l for l in hasil.stdout.splitlines() if 'github.com' not in l))
assert hasil.returncode == 0
CSV = f'{REPO}/outputs/labeling/balanced_3000_v1audited.csv'
assert pathlib.Path(CSV).exists(), 'CSV v1audited tidak ditemukan'

## 2. Muat data & bagi train / val / test
**Teks** (`bert`) dari `processed_bert`; **label** dari CSV audit (menimpa label Mongo).
**Split kanonik** 70/20/10: urut `comment_id` → test 10% → val → train, *stratified*, `seed=42`
— prosedur sama dgn SVM/IndoBERT sehingga **test set identik** (perbandingan adil).

In [ ]:
import pandas as pd, certifi
from pymongo import MongoClient
from sklearn.model_selection import train_test_split

MODEL_NAME = 'indolem/indobertweet-base-uncased'   # pembanding efek weighted-loss: 'indobenchmark/indobert-base-p1'
TAG        = 'v1audited'
LABELS     = ['Negatif', 'Netral', 'Positif']
LABEL2ID   = {l: i for i, l in enumerate(LABELS)}
DB_NAME    = os.environ.get('MONGO_DB_NAME', 'youtube_sentiment')

client = MongoClient(os.environ['MONGO_URI'], tlsCAFile=certifi.where(), serverSelectionTimeoutMS=30000)
client.admin.command('ping'); print('✅ Koneksi MongoDB OK')

sub  = pd.read_csv(CSV); sub['comment_id'] = sub['comment_id'].astype(str)
bert = pd.DataFrame(list(client[DB_NAME]['processed_bert'].find({}, {'_id': 0, 'comment_id': 1, 'bert': 1})))
bert['comment_id'] = bert['comment_id'].astype(str)

df = sub[['comment_id', 'label']].merge(bert, on='comment_id', how='left')   # label dari CSV audit
df['bert']     = df['bert'].fillna('')
df['label_id'] = df['label'].map(LABEL2ID)
assert df['label_id'].notna().all() and (df['bert'].str.len() > 0).all(), 'ada label/teks kosong'

df = df.sort_values('comment_id').reset_index(drop=True)
tmp, df_test     = train_test_split(df,  test_size=0.10,        stratify=df['label_id'],  random_state=42)
df_train, df_val = train_test_split(tmp, test_size=0.20/0.90,   stratify=tmp['label_id'], random_state=42)
print(f'Total {len(df)}  |  train {len(df_train)}  val {len(df_val)}  test {len(df_test)}')
display(df['label'].value_counts().rename('jumlah').to_frame())

In [ ]:
import matplotlib.pyplot as plt
vc = df['label'].value_counts().reindex(LABELS)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(vc.index, vc.values, color=['#C44E52', '#999999', '#55A868'])
for i, v in enumerate(vc.values): ax.text(i, v + 5, str(v), ha='center')
ax.set_title('Distribusi kelas — v1audited'); ax.set_ylabel('jumlah komentar')
plt.tight_layout(); plt.show()

## 3. Tokenisasi (WordPiece)
Transformer butuh teks dipecah jadi **sub-kata**, dipotong maks **128 token**, lalu *padding*.
Contoh hasil tokenisasi ditampilkan di bawah.

In [ ]:
from transformers import AutoTokenizer
import torch
MAX_LEN, SEED = 128, 42
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

contoh = df_train['bert'].iloc[0]
print('Teks  :', contoh)
print('Token :', tok.tokenize(contoh)[:25], '...')

class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=True)
        self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[i]); return item

ds_train = DS(df_train['bert'].astype(str), df_train['label_id'])
ds_val   = DS(df_val['bert'].astype(str),   df_val['label_id'])
ds_test  = DS(df_test['bert'].astype(str),  df_test['label_id'])
print('\n✅ Dataset train/val/test siap')

## 4. Model + Weighted Cross-Entropy
Kepala klasifikasi 3 kelas di atas IndoBERTweet. **Bobot kelas** = terbalik thd frekuensi train
(kelas jarang → bobot besar).

In [ ]:
import numpy as np
from collections import Counter
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, EarlyStoppingCallback, set_seed)
from sklearn.metrics import f1_score, accuracy_score
set_seed(SEED)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={i: l for i, l in enumerate(LABELS)}, label2id=LABEL2ID)

cnt = Counter(df_train['label_id']); n = len(df_train)
class_w = torch.tensor([n / (3 * cnt[i]) for i in range(3)], dtype=torch.float)
print('Bobot kelas (Neg, Net, Pos):', [round(x, 3) for x in class_w.tolist()])

class WeightedTrainer(Trainer):
    """Trainer dgn weighted cross-entropy."""
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=class_w.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def compute_metrics(p):
    pr = np.argmax(p.predictions, axis=1)
    return {'macro_f1': f1_score(p.label_ids, pr, average='macro'),
            'accuracy': accuracy_score(p.label_ids, pr)}

_kw = dict(output_dir='out', num_train_epochs=6, per_device_train_batch_size=16,
           per_device_eval_batch_size=32, learning_rate=2e-5, weight_decay=0.01,
           warmup_ratio=0.1, save_total_limit=2, load_best_model_at_end=True,
           metric_for_best_model='macro_f1', greater_is_better=True,
           seed=SEED, logging_steps=50, report_to='none')
try:
    args = TrainingArguments(eval_strategy='epoch', save_strategy='epoch', **_kw)
except TypeError:
    args = TrainingArguments(evaluation_strategy='epoch', save_strategy='epoch', **_kw)

trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                          compute_metrics=compute_metrics,
                          callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
print('✅ Trainer siap')

## 5. Fine-tuning
*Training loss* turun & *macro-F1 val* naik lalu mendatar. Model akhir = checkpoint val terbaik.

In [ ]:
trainer.train()
print('\nValidasi (model terbaik):', {k: round(v, 4) for k, v in trainer.evaluate().items() if k in ('eval_macro_f1', 'eval_accuracy')})

## 6. Kurva pelatihan per epoch
Memantau overfitting (val-loss naik sementara train-loss turun).

In [ ]:
h = pd.DataFrame(trainer.state.log_history)
ev = h[h['eval_loss'].notna()][['epoch', 'eval_loss', 'eval_accuracy', 'eval_macro_f1']].reset_index(drop=True)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
if 'loss' in h.columns:
    tl = h[h['loss'].notna()]; ax1.plot(tl['epoch'], tl['loss'], marker='.', label='train loss')
ax1.plot(ev['epoch'], ev['eval_loss'], marker='o', label='val loss')
ax1.set_xlabel('epoch'); ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=.3)
ax2.plot(ev['epoch'], ev['eval_accuracy'], marker='o', label='akurasi')
ax2.plot(ev['epoch'], ev['eval_macro_f1'], marker='s', label='macro-F1')
ax2.set_xlabel('epoch'); ax2.set_ylim(0, 1); ax2.set_title('Metrik validasi'); ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()
display(ev.round(4))

## 7. Evaluasi pada test set
Test set belum pernah dilihat model; metrik langsung comparable dgn SVM & IndoBERT.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(ds_test)
y_pred = np.argmax(pred.predictions, axis=1).tolist()
y_true = df_test['label_id'].tolist()

def evaluate(yt, yp, labels=LABELS):
    ids = list(range(len(labels)))
    rep = classification_report(yt, yp, labels=ids, target_names=labels, output_dict=True, zero_division=0)
    return {'accuracy': round(accuracy_score(yt, yp), 4),
            'macro_f1': round(f1_score(yt, yp, average='macro', zero_division=0), 4),
            'weighted_f1': round(f1_score(yt, yp, average='weighted', zero_division=0), 4),
            'per_class': {l: {k: round(rep[l][k], 4) for k in ['precision', 'recall', 'f1-score']} | {'support': int(rep[l]['support'])} for l in labels},
            'confusion_matrix': confusion_matrix(yt, yp, labels=ids).tolist(), 'labels': labels}

m = evaluate(y_true, y_pred)
print(f"🎯  IndoBERTweet [{TAG}]  —  Akurasi {m['accuracy']:.4f}  |  macro-F1 {m['macro_f1']:.4f}")
display(pd.DataFrame(m['per_class']).T[['precision', 'recall', 'f1-score', 'support']])

## 8. Confusion matrix & F1 per kelas

In [ ]:
cm = np.array(m['confusion_matrix']); cmn = cm / cm.sum(axis=1, keepdims=True)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.imshow(cmn, cmap='Purples', vmin=0, vmax=1)
a1.set_xticks(range(3), LABELS); a1.set_yticks(range(3), LABELS)
a1.set_xlabel('Prediksi'); a1.set_ylabel('Aktual'); a1.set_title('Confusion (recall per baris)')
for i in range(3):
    for j in range(3):
        a1.text(j, i, f'{cmn[i,j]:.2f}', ha='center', va='center', color='white' if cmn[i, j] > .5 else 'black')
f1s = [m['per_class'][l]['f1-score'] for l in LABELS]
a2.bar(LABELS, f1s, color=['#C44E52', '#999999', '#55A868'])
for i, v in enumerate(f1s): a2.text(i, v + .01, f'{v:.3f}', ha='center')
a2.set_ylim(0, 1); a2.set_title('F1 per kelas')
plt.tight_layout(); plt.show()
fig.savefig('indobertweet_v1audited_test_confusion.png', dpi=120)


## 9. Contoh prediksi (benar & salah)
Kasus SALAH paling-yakin = kandidat label yang perlu ditinjau.

In [ ]:
logits = np.asarray(pred.predictions, dtype=float)
e = np.exp(logits - logits.max(axis=1, keepdims=True))
conf = (e / e.sum(axis=1, keepdims=True)).max(axis=1)
p = pd.DataFrame({'teks': df_test['bert'].astype(str).str.slice(0, 80).to_numpy(),
                  'asli': df_test['label'].to_numpy(),
                  'prediksi': [LABELS[i] for i in y_pred],
                  'keyakinan': np.round(conf, 3)})
p['benar'] = p['asli'] == p['prediksi']
print(f"Benar {p['benar'].sum()} / {len(p)}  ({p['benar'].mean():.1%})")
print('\n— Contoh SALAH (paling yakin) —')
display(p[~p['benar']].sort_values('keyakinan', ascending=False).head(8))
print('— Contoh BENAR —')
display(p[p['benar']].head(6))

## 10. Perbandingan 3 model (v1audited)
IndoBERTweet vs SVM & IndoBERT (artefak dari repo, test set identik).

In [ ]:
import json
rep_dir = f'{REPO}/outputs/reports'
def baca(f):
    pth = pathlib.Path(rep_dir, f)
    if pth.exists():
        t = json.load(open(pth))['test']; return t['accuracy'], t['macro_f1']
    return None
rows = []
for nama, f in [('SVM', 'svm_v1audited_metrics.json'),
                ('IndoBERT', 'indobert_v1audited_metrics.json')]:
    r = baca(f)
    if r: rows.append({'model': nama, 'akurasi': r[0], 'macro_f1': r[1]})
rows.append({'model': 'IndoBERTweet', 'akurasi': m['accuracy'], 'macro_f1': m['macro_f1']})
cmp = pd.DataFrame(rows); display(cmp)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
bars = ax.bar(cmp['model'], cmp['akurasi'], color=['#4C72B0', '#DD8452', '#8172B3'][:len(cmp)], width=.55)
bi = int(cmp['akurasi'].idxmax())
for i, (b, v) in enumerate(zip(bars, cmp['akurasi'])):
    ax.text(b.get_x() + b.get_width()/2, v + .006, f'{v:.4f}' + ('  ★' if i == bi else ''), ha='center')
ax.set_ylim(0, .95); ax.set_ylabel('Akurasi (test, n=300)'); ax.set_title('Perbandingan Model — v1audited')
plt.tight_layout(); plt.show()

## 11. Simpan & unduh artefak

In [ ]:
json.dump({'model': MODEL_NAME, 'dataset': TAG, 'test': m},
          open('indobertweet_v1audited_metrics.json', 'w'), ensure_ascii=False, indent=2)
full = pd.DataFrame({'comment_id': df_test['comment_id'].to_numpy(), 'text': df_test['bert'].to_numpy(),
                     'label_asli': df_test['label'].to_numpy(), 'prediksi': [LABELS[i] for i in y_pred],
                     'benar': p['benar'].to_numpy(), 'keyakinan': np.round(conf, 4)})
full.to_csv('indobertweet_v1audited_predictions.csv', index=False)
print('✅ Tersimpan: indobertweet_v1audited_metrics.json / _predictions.csv / _test_confusion.png')
try:
    from google.colab import files
    files.download('indobertweet_v1audited_metrics.json')
    files.download('indobertweet_v1audited_predictions.csv')
except Exception as ex:
    print('Unduh manual dari panel Files. Detail:', ex)